# De-confound + Downstream-QA Sweep (run everything top to bottom)

One self-contained notebook for the two missing experiments of the RAPTOR comparison study
(protocol: `docs/2026-07-21-raptor-comparison-runbook.md`):

- **Cell A — de-confound** (free): rebuild the tree sweep with an *extractive* summarizer.
  Settles whether the tree's low coverage was the hierarchy or gpt-oss's bloated summaries.
- **Cell B — QA sweep** (~$0.10–0.20): answer every question *from its retrieved context*
  (flat and tree, same QA model) and grade with answer-F1. The paper's new headline data.
- **Cell C — analysis**: answer-F1 by leaf size and by question type (lookup / multi-hop / aggregation).

The old flat/tree *coverage* sweeps are **not** here — those results are already committed
(`experiments/results/sweep_{flat,tree}_n50.json`).

### ⚠️ Run it as a Kaggle BATCH job, not a draft
`/kaggle/working` is **deleted when a draft (Interactive) session ends** — outputs *and* the
LLM cache. A disconnect therefore costs the whole run, not a resume. Cell 2 refuses to
proceed in a draft session for exactly this reason.

1. **Settings → Internet → On**, **Accelerator → GPU T4**
2. **Add-ons → Secrets** → add `OPENROUTER_API_KEY` (paid credit; the run spends ~$0.10–0.20)
3. **Save Version → "Save & Run All (Commit)"** → close the tab. 12 h limit; the finished
   version's Output tab holds the JSONs.

Colab works too: `run_dir()` mounts Drive, and *that* path really does resume across
session deaths.

### Time
| cell | time | cost |
|---|---|---|
| 1–3 setup | ~10 min | $0 |
| A de-confound | ~30–60 min | $0 |
| B flat QA | ~30–45 min | ~$0.05 |
| B tree QA | ~1–2 h warm cache / ~4 h cold | ~$0.05 / ~$0.15 |
| C analysis | seconds | $0 |

**Resume, honestly:** re-running a cell skips completed `(doc, size)` pairs and reuses
DiskCache'd LLM calls — *provided `RUN_DIR` still exists*. On Drive or a committed Kaggle
version it does; in a Kaggle draft it does not.

When done: run the last cell, download `sweep_outputs.zip`, drop the three JSONs into
`experiments/results/` in the repo.


In [ ]:
# 1) Code + preflight. Heavy deps are imported LAZILY inside RAPTOR, so a light
#    import proves nothing — check them for real or the sweep dies confusingly.
!git clone -b ckraptor https://github.com/MissLostCodes/raptor.git
%cd raptor
!pip install -q -r requirements-colab.txt

from experiments.granularity_sweep import run_sweep

import importlib
missing = []
for mod in ['torch', 'sentence_transformers', 'umap', 'faiss', 'sklearn', 'tiktoken']:
    try:
        importlib.import_module(mod)
    except Exception as e:
        missing.append(f'{mod:<22} {type(e).__name__}: {e}')
if missing:
    print('PREFLIGHT FAILED ❌\n')
    for m in missing:
        print('   ', m)
    print('\nKaggle: Internet On, then Run -> Restart Session and re-run.')
    raise SystemExit('preflight failed')

print('heavy stack present ✅')


In [ ]:
# 2) Config + secrets + persisted output paths.
N_DOCS  = 50                             # the paper's H1/H2 cohort
SIZES   = [50, 100, 150, 200, 300, 400]  # the paper's sweep grid
BUDGET  = 2000                           # retrieval token budget (paper's setting)

# PAID model id: no ':free' suffix (free tier is capped at 1000 req/day).
MODEL = 'openai/gpt-oss-120b'

import os

def api_key():
    try:
        from google.colab import userdata
        return userdata.get('OPENROUTER_API_KEY')
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('OPENROUTER_API_KEY')
    except Exception:
        pass
    return os.environ.get('OPENROUTER_API_KEY')

key = api_key()
if not key:
    raise SystemExit('Need OPENROUTER_API_KEY (Kaggle: Add-ons -> Secrets). '
                     'Cell A is free, but Cell B needs it for QA calls.')
os.environ['OPENROUTER_API_KEY'] = key

def run_dir():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        return '/content/drive/MyDrive/raptor_runs'
    except Exception:
        pass
    if os.path.isdir('/kaggle/working'):
        return '/kaggle/working/raptor_runs'
    return 'raptor_runs'

RUN_DIR = run_dir()
os.makedirs(RUN_DIR, exist_ok=True)

# STOP if the run has nowhere durable to write. A Kaggle *draft* (Interactive)
# session wipes /kaggle/working the moment it dies -- outputs AND the LLM cache,
# so "just re-run it" is a full-price restart, not a resume. Only a committed
# batch run (Save Version -> Save & Run All) keeps its output. Learned the hard
# way: an 8 h overnight draft run left an empty directory.
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Interactive':
    raise SystemExit(
        'Kaggle DRAFT session: /kaggle/working does NOT survive a disconnect.\n'
        'Use one of:\n'
        '  * Save Version -> "Save & Run All (Commit)"  -- headless, 12 h limit,\n'
        '    output persists as a notebook version you can download; closing the\n'
        '    browser is safe. Set GPU T4 + Internet ON in Settings first.\n'
        '  * Colab with Drive mounted -- run_dir() picks /content/drive and every\n'
        '    cell then genuinely resumes across session deaths.\n'
        'To override anyway (short test run, results expendable):\n'
        '    os.environ["KAGGLE_KERNEL_RUN_TYPE"] = "Batch"'
    )

# SEPARATE file per mode/run: the resume key is (doc_id, size) and carries no mode.
# run_sweep also refuses to resume a QA run from a retrieval-only cache (and tree
# from flat), but distinct names are the real fix.
OFFLINE_OUT = os.path.join(RUN_DIR, f'sweep_tree_offline_n{N_DOCS}.json')
FLAT_QA_OUT = os.path.join(RUN_DIR, f'sweep_flat_qa_n{N_DOCS}.json')
TREE_QA_OUT = os.path.join(RUN_DIR, f'sweep_tree_qa_n{N_DOCS}.json')
print('RUN_DIR     =', RUN_DIR)
print('OFFLINE_OUT =', OFFLINE_OUT)
print('FLAT_QA_OUT =', FLAT_QA_OUT)
print('TREE_QA_OUT =', TREE_QA_OUT)


In [ ]:
# 3) Load the QASPER cohort (same split + loader the paper uses).
from experiments.datasets import get_loader
import statistics

docs = get_loader('qasper').load(limit=N_DOCS)
lens = [len(d.text.split()) for d in docs]
nq = sum(len(d.questions) for d in docs)
print(f'{len(docs)} docs | {nq} questions | median {statistics.median(lens):.0f} words')


In [ ]:
# A) DE-CONFOUND (free, ~30-60 min): same tree sweep, but summaries are an
#    extractive slice — no API calls, hierarchy shape preserved. Two readings:
#      * extractive tree recovers toward flat -> gpt-oss summaries were the bottleneck
#      * extractive tree stays low            -> the hierarchy itself underperforms
#        (on the coverage metric; Cell B then checks ANSWER quality)
#    Resumable: re-run after a timeout; completed (doc, size) pairs are skipped.
from experiments.tree_cost_dryrun import OfflineSummarizer
from tqdm.auto import tqdm

tree_offline = run_sweep(
    docs, SIZES, budget=BUDGET, out_path=OFFLINE_OUT,
    summarization_model=OfflineSummarizer(),      # $0, no network
    progress=lambda m: tqdm.write(m),
)
print(f'\noffline tree: {len(tree_offline)} (doc, size) records -> {OFFLINE_OUT}')


In [ ]:
# A2) GO/NO-GO readout: flat vs tree(gpt-oss) vs tree(extractive), coverage by size.
#     The first two are the committed July-16 results shipped inside the cloned repo
#     (same 50 docs), so this cell is free and immediate.
import json, statistics as st
from collections import defaultdict

def cov_by_size(recs):
    d = defaultdict(list)
    for r in recs:
        c = r.get('mean_evidence_coverage')
        if c is not None:
            d[r['size']].append(c)
    return {s: st.mean(v) for s, v in d.items()}

flat_ref = json.load(open('experiments/results/sweep_flat_n50.json'))
tree_ref = json.load(open('experiments/results/sweep_tree_n50.json'))
f, t, o = cov_by_size(flat_ref), cov_by_size(tree_ref), cov_by_size(tree_offline)

print(f"{'size':>5} | {'flat':>6} | {'tree gpt-oss':>12} | {'tree extractive':>15}")
print('-' * 49)
for s in sorted(f):
    print(f'{s:>5} | {f[s]:>6.3f} | {t.get(s, float("nan")):>12.3f} | {o.get(s, float("nan")):>15.3f}')

print('\nReading: extractive column near FLAT column  -> summarizer-bound (gpt-oss was the problem);')
print('         extractive column near GPT-OSS column -> hierarchy itself scores lower on coverage.')


In [ ]:
# B) QA SWEEP (the new finding, ~$0.10-0.20): answer each question FROM its
#    retrieved context, grade with answer-F1 vs gold. Same qa_model for flat and
#    tree -> a fair answer-quality comparison (not just retrieval coverage).
#    Tree summaries reuse the DiskCache when warm; a cold cache re-pays ~$0.09.
#    Both sweeps resumable; QA answers are also cached, so restarts cost ~nothing.
from experiments.config import ExperimentConfig
from experiments.models import build_models
from tqdm.auto import tqdm

cfg = ExperimentConfig(model=MODEL, cache_dir=os.path.join(RUN_DIR, '.llm_cache'))
summarization_model, qa_model = build_models(cfg)   # share one DiskCache

flat_qa = run_sweep(docs, SIZES, budget=BUDGET, out_path=FLAT_QA_OUT,
                    qa_model=qa_model, progress=lambda m: tqdm.write(m))
print(f'\nflat+QA done: {len(flat_qa)} records')

tree_qa = run_sweep(docs, SIZES, budget=BUDGET, out_path=TREE_QA_OUT,
                    summarization_model=summarization_model, qa_model=qa_model,
                    progress=lambda m: tqdm.write(m))
print(f'tree+QA done: {len(tree_qa)} records')


In [ ]:
# C) Where does the tree actually help on ANSWERS? answer-F1 by leaf size and by
#    question type (m = # gold-evidence paragraphs: 1 lookup, 2 multi-hop, >=3 aggregation).
#    Reads Cell B's output files, so Cell B must have run (it writes after each doc).
import os, json, statistics as st
from collections import defaultdict

for p in (FLAT_QA_OUT, TREE_QA_OUT):
    if not os.path.exists(p):
        raise SystemExit(f'{p} not found -> run Cell B first (it writes this file '
                         f'after each document, so even a partial B is enough).')

flat_qa = json.load(open(FLAT_QA_OUT))
tree_qa = json.load(open(TREE_QA_OUT))

def bucket(m):
    return '1 lookup' if m == 1 else ('2 multi-hop' if m == 2 else '>=3 aggregation')

def by_size_f1(recs):
    d = defaultdict(list)
    for r in recs:
        if r.get('mean_answer_f1') is not None:
            d[r['size']].append(r['mean_answer_f1'])
    return {s: st.mean(v) for s, v in d.items()}

def by_type_f1(recs):
    d = defaultdict(list)
    for r in recs:
        for pq in r.get('per_question', []):
            if 'answer_f1' in pq and pq.get('m'):
                d[bucket(pq['m'])].append(pq['answer_f1'])
    return {k: (st.mean(v), len(v)) for k, v in d.items()}

fs, ts = by_size_f1(flat_qa), by_size_f1(tree_qa)
print('answer-F1 by leaf size:   size    flat    tree')
for s in sorted(fs):
    print(f'   {s:>4}   {fs[s]:.3f}   {ts.get(s, float("nan")):.3f}')

ft, tt = by_type_f1(flat_qa), by_type_f1(tree_qa)
print('\nanswer-F1 by question type (all sizes pooled):')
for k in ['1 lookup', '2 multi-hop', '>=3 aggregation']:
    fv = ft.get(k, (float('nan'), 0))
    tv = tt.get(k, (float('nan'), 0))
    print(f'   {k:<16} flat {fv[0]:.3f} (n={fv[1]})   tree {tv[0]:.3f} (n={tv[1]})')


In [ ]:
# D) Collect outputs. Download sweep_outputs.zip (right panel -> Output), then drop
#    the three JSONs into experiments/results/ in the repo and say "done".
import os, zipfile

zp = os.path.join(RUN_DIR, 'sweep_outputs.zip')
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [OFFLINE_OUT, FLAT_QA_OUT, TREE_QA_OUT]:
        status = 'OK  ' if os.path.exists(p) else 'MISS'
        print(status, p)
        if os.path.exists(p):
            z.write(p, os.path.basename(p))
print('\nzipped ->', zp)
